In [1]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline


from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import classification_report, accuracy_score, f1_score

In [2]:
data = pd.read_excel("../data/job_classification.ods", engine = "odf", dtype = str)

data.head()

,title,location,description,function,industry,career_level
0,Technical Professional Lead - Process,"Houston, TX","Responsible for the study, design, and specifi...",production_manufacturing,Machinery and Industrial Facilities Engineering,senior_specialist_or_project_manager
1,Cnslt - Systems Eng- Midrange 1,"Seattle, WA","Participates in design, development and implem...",information_technology_telecommunications,Financial Services,senior_specialist_or_project_manager
2,SharePoint Developers and Solution Architects,"Dallas, TX",We are currently in need of Developers who can...,consulting,IT Consulting,senior_specialist_or_project_manager
3,Business Information Services - Strategic Acco...,North Carolina,Experian is seeking an experienced Account Exe...,sales,"Security, Risk, Restructuring Consulting",senior_specialist_or_project_manager
4,Strategic Development Director (procurement),"Austin, TX",Â Want to join a world-class global procuremen...,procurement_materials_logistics,Information Technology,bereichsleiter


In [3]:
data = data.dropna(axis = 0)
data.shape

(8073, 6)

In [4]:
def filler_location(location):
    result = re.findall("\\,\\s[A-Z]{2}$", location)
    if len(result) > 0:
        return result[0][2:]
    else:
        return location


data["location"] = data["location"].apply(filler_location)

In [5]:
target = "career_level"
x = data.drop(target, axis = 1)
y = data[target]

In [11]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 42, stratify = y)

In [12]:
transformers = ColumnTransformer(transformers = [
    ("title", TfidfVectorizer(stop_words="english"), "title"),
    ("location", OneHotEncoder(handle_unknown="ignore"), ["location"]),
    ("description", TfidfVectorizer(stop_words="english", ngram_range=(1,2)), "description"),
    ("function", OneHotEncoder(handle_unknown="ignore"), ["function"]),
    ("industry", TfidfVectorizer(stop_words="english"), "industry")
])

In [13]:
logistic_model = Pipeline(steps = [
    ("transformers", transformers),
    ("classifier", LogisticRegression())
])

In [14]:
logistic_model.fit(x_train, y_train)
y_logistic_predict = logistic_model.predict(x_test)
print(classification_report(y_test, y_logistic_predict))

                                        precision    recall  f1-score   support

                        bereichsleiter       0.51      0.40      0.45       192
         director_business_unit_leader       0.80      0.29      0.42        14
                   manager_team_leader       0.66      0.65      0.66       534
managing_director_small_medium_company       0.00      0.00      0.00         1
  senior_specialist_or_project_manager       0.84      0.91      0.87       868
                            specialist       0.00      0.00      0.00         6

                              accuracy                           0.75      1615
                             macro avg       0.47      0.37      0.40      1615
                          weighted avg       0.74      0.75      0.74      1615



C:\Users\GIGABYTE\PycharmProjects\Job-Level-Classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\GIGABYTE\PycharmProjects\Job-Level-Classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\GIGABYTE\PycharmProjects\Job-Level-Classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control 

In [15]:
svm_model = Pipeline(steps=[
    ("transformers", transformers),
    ("classifier", SVC())
])

In [16]:
svm_model.fit(x_train, y_train)
y_svm_predict = svm_model.predict(x_test)
print(classification_report(y_test, y_svm_predict))

                                        precision    recall  f1-score   support

                        bereichsleiter       0.66      0.30      0.41       192
         director_business_unit_leader       0.75      0.21      0.33        14
                   manager_team_leader       0.67      0.69      0.68       534
managing_director_small_medium_company       0.00      0.00      0.00         1
  senior_specialist_or_project_manager       0.82      0.92      0.87       868
                            specialist       0.00      0.00      0.00         6

                              accuracy                           0.76      1615
                             macro avg       0.48      0.35      0.38      1615
                          weighted avg       0.75      0.76      0.74      1615



C:\Users\GIGABYTE\PycharmProjects\Job-Level-Classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\GIGABYTE\PycharmProjects\Job-Level-Classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\GIGABYTE\PycharmProjects\Job-Level-Classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control 

In [17]:
rf_model = Pipeline(steps=[
    ("transformers", transformers),
    ("classifier", RandomForestClassifier(random_state=42))
])

In [18]:
rf_model.fit(x_train, y_train)
y_rf_predict = rf_model.predict(x_test)
print(classification_report(y_test, y_rf_predict))

                                        precision    recall  f1-score   support

                        bereichsleiter       0.67      0.05      0.10       192
         director_business_unit_leader       1.00      0.29      0.44        14
                   manager_team_leader       0.62      0.52      0.56       534
managing_director_small_medium_company       0.00      0.00      0.00         1
  senior_specialist_or_project_manager       0.72      0.96      0.82       868
                            specialist       0.00      0.00      0.00         6

                              accuracy                           0.69      1615
                             macro avg       0.50      0.30      0.32      1615
                          weighted avg       0.68      0.69      0.64      1615



C:\Users\GIGABYTE\PycharmProjects\Job-Level-Classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\GIGABYTE\PycharmProjects\Job-Level-Classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\GIGABYTE\PycharmProjects\Job-Level-Classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control 

In [19]:
results = pd.DataFrame({
    "Experiment": [
        "Logistic Regression",
        "Linear SVM",
        "Random Forest"
    ],
    "Accuracy": [
        accuracy_score(y_test, y_logistic_predict),
        accuracy_score(y_test, y_svm_predict),
        accuracy_score(y_test, y_rf_predict)
    ],
    "Macro F1": [
        f1_score(y_test, y_logistic_predict, average="macro"),
        f1_score(y_test, y_svm_predict, average="macro"),
        f1_score(y_test, y_rf_predict, average="macro")
    ],
    "Weighted F1": [
        f1_score(y_test, y_logistic_predict, average="weighted"),
        f1_score(y_test, y_svm_predict, average="weighted"),
        f1_score(y_test, y_rf_predict, average="weighted")
    ]
})

results

,Experiment,Accuracy,Macro F1,Weighted F1
0,Logistic Regression,0.753560,0.400277,0.744582
1,Linear SVM,0.760372,0.381785,0.742442
2,Random Forest,0.692879,0.320989,0.643164
